In [2]:
# %pip install -U langgraph langchain langchain-google-genai langchain-community langchain-chroma pypdf python-dotenv


In [3]:
from typing import TypedDict
import os
from dotenv import load_dotenv


from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings

from langchain_chroma import Chroma

from langgraph.graph import StateGraph, START, END

In [4]:
load_dotenv()

True

In [5]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)

In [6]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [7]:
loader = PyPDFLoader("ML.pdf")

documents = [] # ek baar initialize kiya, aur har loader cell mein
documents.extend(loader.load()) # use kiya — taaki teeno PDFs ka data ek hi list mein jama ho.

print("Number of pages:", len(documents))

Number of pages: 9


In [8]:
loader = PyPDFLoader("HR_Policy.pdf")

documents.extend(loader.load())

print("Number of pages:", len(documents))

Number of pages: 14


In [9]:
loader = PyPDFLoader("IT_Security_Policy.pdf")

documents.extend(loader.load())

print("Number of pages:", len(documents))

Number of pages: 17


In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

In [11]:
chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 56


In [12]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="chroma_db"
)

In [13]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [14]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    collection_name="chroma_db",
    embedding_function=embeddings
)

print("Chroma DB created successfully!")

Chroma DB created successfully!


In [15]:
class State(TypedDict):
    question: str
    documents: list
    answer: str

In [16]:
def retrieve_node(state: State):

    question = state["question"]

    documents = retriever.invoke(question)

    return {
        "documents": documents
    }

In [17]:
def generate_node(state: State):

    question = state["question"]

    documents = state["documents"]

    context = "\n\n".join(
        doc.page_content
        for doc in documents
    )
#temlate ""ai - instruction 
    prompt = f"""
    Answer the question using only the context below.

    Context:
    {context}

    Question:
    {question}

    Answer only from the context.

If the answer is not present in the context, say:
"I don't know based on the provided document."

Format the response as:
- Short introduction
- Main answer
- Important points as bullet points

Return plain text only.
Do not return JSON.
Do not return a Python list.
    
    If the answer is not present in the context,
    say "I don't know based on the provided document."
    """

    response = llm.invoke(prompt)

    content = response.content
    if isinstance(content, list):
        answer_text = "".join([part.get("text", "") if isinstance(part, dict) else str(part) for part in content])
    else:
        answer_text = str(content)

    return {
        "answer": answer_text
    }


In [18]:
graph = StateGraph(State)

In [19]:
graph.add_node("retrieve", retrieve_node)

graph.add_node("generate", generate_node)

In [20]:
graph.add_edge(START, "retrieve")

graph.add_edge("retrieve", "generate")

graph.add_edge("generate", END)

In [21]:
app = graph.compile()

In [22]:
result = app.invoke({
    "question": "What is supervisied learning",
    "documents": [],
    "answer": ""
})

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [23]:
print("Question:")
print(result["question"])

print("\nAnswer:")
print(result["answer"])

Question:
What is supervisied learning

Answer:
Short introduction:
Based on the provided document, here is the definition and details of supervised learning.

Main answer:
Supervised learning is a type of machine learning where the training data contains both input features and a known target or output, which is referred to as labeled data.

Important points:
* The training data includes both input features and a known target/output (labeled data).
* Common algorithms used in supervised learning are Linear Regression, Logistic Regression, Decision Tree, Random Forest, SVM, and KNN.
